# Multi-Agent AI Music Producer - Colab Test

This notebook runs the full workflow with HuggingFace models on Colab GPU.

**Requirements:** Run with GPU runtime (T4 or better)
- Runtime → Change runtime type → T4 GPU

In [ ]:
# 1. Check GPU availability
!nvidia-smi

In [ ]:
# 2. Mount Google Drive and clone repo (persists between sessions!)
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive

# Clone repo (only first time - comment out after)
!git clone https://github.com/YOUR_USERNAME/multi_agent_ai_music_producer.git

# If already cloned, just pull latest:
# %cd /content/drive/MyDrive/multi_agent_ai_music_producer && git pull

In [ ]:
# 3. Go to project directory
%cd /content/drive/MyDrive/multi_agent_ai_music_producer

In [ ]:
# 4. Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate bitsandbytes
!pip install -q langgraph pydantic pyyaml scipy soundfile
!pip install -q -e .

In [ ]:
# 5. Set HuggingFace token
import os
os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"  # Replace with your token

In [ ]:
# 6. Test HuggingFace provider
from src.llm.huggingface_provider import HuggingFaceProvider
from src.llm.base import LLMMessage

provider = HuggingFaceProvider(
    model="Qwen/Qwen2.5-7B-Instruct",
    token=os.environ["HF_TOKEN"],
    load_in_4bit=True,  # Required for T4 GPU!
    temperature=0.7,
    max_tokens=512,
)

messages = [LLMMessage(role="user", content="Say hello in 3 words")]
response = await provider.generate(messages)
print(f"Response: {response.content}")

In [ ]:
# 7. Setup workflow
from pathlib import Path
from unittest.mock import patch
import numpy as np
import soundfile as sf

from src.config import Settings, LLMConfig, GenerationConfig, AudioConfig, LoggingConfig
from src.logging.logger import MusicProducerLogger, LogLevel
from src.graph.workflow import MusicProducerGraph

output_dir = Path("/content/output")
output_dir.mkdir(exist_ok=True)

# Mock audio generator
class MockAudioGenerator:
    def __init__(self, sample_rate=32000):
        self.sample_rate = sample_rate
        self.call_count = 0
    
    def generate(self, prompt, duration_sec, output_path, **kwargs):
        self.call_count += 1
        t = np.linspace(0, duration_sec, int(self.sample_rate * duration_sec))
        audio = 0.3 * np.sin(2 * np.pi * (220 + self.call_count * 55) * t)
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        sf.write(output_path, audio, self.sample_rate)
        return {"success": True, "path": output_path}

mock_audio = MockAudioGenerator()

settings = Settings(
    llm=LLMConfig(
        provider="huggingface",
        model="Qwen/Qwen2.5-7B-Instruct",
        temperature=0.7,
        max_tokens=1024,
    ),
    generation=GenerationConfig(max_retries=1, default_segment_duration=5.0),
    audio=AudioConfig(sample_rate=32000),
    logging=LoggingConfig(level="DEBUG", output_dir=str(output_dir / "logs")),
)

logger = MusicProducerLogger(
    run_id="colab_run",
    output_dir=str(output_dir),
    level=LogLevel.DEBUG,
    console_output=True,
)

print(f"Ready! Model: {settings.llm.model}")

In [ ]:
# 8. Run the workflow!
with patch("src.tools.audio_generation.generate_segment") as mock_gen:
    mock_gen.side_effect = lambda prompt, duration_sec, output_path, **kw: \
        mock_audio.generate(prompt, duration_sec, output_path, **kw)
    
    graph = MusicProducerGraph(settings=settings, logger=logger)
    graph.build()
    
    print("\n" + "="*50)
    print("Starting Music Production")
    print("="*50 + "\n")
    
    result = graph.invoke(
        user_prompt="Create a chill lofi hip-hop beat with jazzy piano and mellow drums",
        reference_paths=[],
    )
    
    print("\n" + "="*50)
    print("Done!")
    print(f"Phase: {result.get('phase')}")
    print(f"Plan: {result.get('track_plan')}")

In [ ]:
# 9. Play the audio (if generated)
from IPython.display import Audio
if result.get('final_track_path'):
    Audio(result['final_track_path'])